# Libraries


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Variables

**Duty cycle**

In [ ]:
#10797420 so 20 % 50 TODO check this
timeSleepSeconds = (20 % 50 + 5) / 10.0 
timeSleepMicroseconds = timeSleepSeconds * 1000000
print(timeSleepMicroseconds, "microseconds")

**Battery**

In [ ]:
#10797420 so 7420 TODO check this
JBattery= 7420%5000+15000
print(JBattery, "Joule")

# Dataset Analysis

**Structure of the dataset**

In [ ]:
t_boot = 0.000941
t_sensor = 0.000849
t_wifi_init = 0.186580
t_tx = 0.054831
t_sleep_prep = 0.000258
t_sleep = 2.5

df_sleep = pd.read_csv('data/deep_sleep.csv')
df_sender = pd.read_csv('data/sender.csv')
df_sensor = pd.read_csv('data/sensor-read.csv')

**Description of the dataset**

In [ ]:
print(df_sleep.describe())
print(df_sender.describe())
print(df_sensor.describe())

From this graph it's possible to distinguish when the ESP32 is in a particular state.

In [ ]:
df_sleep.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Deep Sleep)",
    kind="line",
    legend=False,
)

In [ ]:
df_sender.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Sender)",
    kind="line",
    legend=False,
)

In [ ]:
df_sensor.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Sensor)",
    kind="line",
    legend=False,
)

# Average Power Consumption

Computing the average power consuption during each functional state requires filtering the datapoints according to empirical observation of the plotted data.

| deep sleep | idle              | WiFi on  |
|------------|-------------------|----------|
| < 100 mW   | >300 mW, < 500 mW | > 600 mW | 

Computing the average power consuption during the **deep sleep** state, it's filtered all the datapoints below the 100W

In [ ]:
p_sensor_mean = df_sensor['Data'].mean() 
p_sensor_idle = df_sensor['Data'][df_sensor['Data'] < 300].mean() # filter to just the idle baseline
p_sensor_active = df_sensor['Data'][df_sensor['Data'] > 300].mean() # filter to just the active baseline
p_tx = df_sender['Data'].max() #in main.io we transmit at 19,5 dBm, which is the max from the slides
p_sleep = df_sleep['Data'][df_sleep['Data'] < 100].mean() # filter to just the deep sleep baseline
p_boot = 340   #from the grpahs    #TODO extract from deep sleep data?
p_wifi_on = df_sender['Data'][df_sender['Data'] < 610].mean() # filter to just the wifi init
p_wifi_2dBm = df_sender['Data'][df_sender['Data'] > 610][df_sender['Data'] < 630].mean() # from 610 to 630 is 2 dBm
p_wifi_19dBm = df_sender['Data'][df_sender['Data'] > 635].mean() #from 635 19.5 dBm 



In [ ]:
import numpy as np

# energy calculations
e_boot = (p_boot / 1000) * t_boot
e_sensor = (p_sensor_active / 1000) * t_sensor
e_wifi_on = (p_wifi_on / 1000) * t_wifi_init
e_tx = (p_tx / 1000) * t_tx
e_sleep_prep = (p_wifi_on / 1000) * t_sleep_prep
e_sleep = (p_sleep / 1000) * t_sleep

e_total = e_boot + e_sensor + e_wifi_on + e_tx + e_sleep_prep + e_sleep
print(f"Total Energy per cycle: {e_total:.6f} Joules")

# battery life estimation
total_time_per_cycle = t_boot + t_sensor + t_wifi_init + t_tx + t_sleep_prep + t_sleep
cycles = JBattery / e_total
total_time_seconds = cycles * total_time_per_cycle
days = total_time_seconds / (60 * 60 * 24)

print(f"Estimated Battery Life: {days:.2f} days")

# graphing power consumption over one cycle
times = [0, t_boot, t_boot+t_sensor, t_boot+t_sensor+t_wifi_init, 
         t_boot+t_sensor+t_wifi_init+t_tx, total_time_per_cycle - t_sleep, total_time_per_cycle]

powers = [p_boot, p_sensor_active, p_wifi_on, p_tx, p_wifi_on, p_sleep]

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(times[:-1], powers, where='post', color='b', linewidth=2)
ax.fill_between(times[:-1], powers, step="post", color='b', alpha=0.2)

ax.set_title('Estimated Power Consumption Over One Cycle')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Power (mW)')
ax.grid(True, which="both", ls="--", alpha=0.5)

plt.show()